# 🎥 Avenue Dataset — Abnormal Event Detection + Explainable AI

> **Pipeline:** MobileNetV2 → LSTM (Attention) → Sigmoid → XAI Explanations

> **XAI Methods:** Grad-CAM | Saliency Maps | SHAP | LIME | Attention Weights | LSTM Gate Analysis | t-SNE | Threshold Analysis

---

### CELL 1 — Install & Import Libraries (+ XAI libraries)

In [1]:
!pip install tensorflow opencv-python matplotlib scikit-learn tqdm numpy pandas seaborn
!pip install shap lime tf-keras-vis

import os, cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras import Model, Input, backend as K
from tensorflow.keras.layers import (LSTM, Dense, Dropout, GlobalAveragePooling2D,
                                      BatchNormalization, Attention, Multiply, Lambda)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                              classification_report)
from sklearn.model_selection import train_test_split

# XAI Libraries
import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow : {tf.__version__}")
print(f"OpenCV     : {cv2.__version__}")
print(f"GPU        : {tf.config.list_physical_devices('GPU')}")

Defaulting to user installation because normal site-packages is not writeable


Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for shap from https://files.pythonhosted.org/packages/a5/8e/cee1ee136a4e54fe2fbb63a60d72d7c25e21a4ffe6aa05779cab7669cb31/shap-0.51.0-cp311-cp311-win_amd64.whl.metadata
     ---------------------------------------- 0.0/275.7 kB ? eta -:--:--
     - -------------------------------------- 10.2/275.7 kB ? eta -:--:--
     ---- -------------------------------- 30.7/275.7 kB 330.3 kB/s eta 0:00:01
     --------- --------------------------- 71.7/275.7 kB 563.7 kB/s eta 0:00:01
     ---------------- ------------------- 122.9/275.7 kB 722.1 kB/s eta 0:00:01
     ----------------------------------- -- 256.0/275.7 kB 1.2 MB/s eta 0:00:01
     -------------------------------------- 275.7/275.7 kB 1.1 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Obtaining dependency information for tf-keras-vis from https://files

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\awati\\AppData\\Roaming\\Python\\Python311\\site-packages\\~umpy.libs\\libopenblas64__v0.3.23-293-gc2f4bdbb-gcc_10_3_0-2bde3a66a51006b2b53eb373ff767a3f.dll'
Check the permissions.



ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

### CELL 2 — Configuration & Paths

In [ ]:
BASE_DIR        = r"C:\Users\awati\Avenue_Dataset 3\Avenue Dataset"
TRAIN_VIDEO_DIR = os.path.join(BASE_DIR, "training_videos")
TEST_VIDEO_DIR  = os.path.join(BASE_DIR, "testing_videos")
XAI_DIR         = os.path.join(BASE_DIR, "xai_outputs")          # NEW: XAI output folder
os.makedirs(XAI_DIR, exist_ok=True)

IMG_HEIGHT   = 224
IMG_WIDTH    = 224
FRAME_SKIP   = 5
SEQ_LENGTH   = 16
STEP_SIZE    = 8
BATCH_SIZE   = 8
EPOCHS       = 30
LEARNING_RATE = 1e-4
VAL_SPLIT    = 0.15
THRESHOLD    = 0.5
CACHE_DIR    = os.path.join(BASE_DIR, "feature_cache")
MODEL_PATH   = os.path.join(BASE_DIR, "mobilenet_lstm_abnormal.keras")
os.makedirs(CACHE_DIR, exist_ok=True)

for name, path in [("Base", BASE_DIR), ("Train", TRAIN_VIDEO_DIR),
                   ("Test", TEST_VIDEO_DIR), ("XAI", XAI_DIR)]:
    print(f"{'✅ Found' if os.path.exists(path) else '❌ MISSING'}  [{name}]  {path}")

train_videos = sorted([f for f in os.listdir(TRAIN_VIDEO_DIR)
                       if f.endswith(('.avi','.mp4','.AVI','.MP4'))])
test_videos  = sorted([f for f in os.listdir(TEST_VIDEO_DIR)
                       if f.endswith(('.avi','.mp4','.AVI','.MP4'))])
print(f"\nTraining videos : {len(train_videos)}  |  Testing videos : {len(test_videos)}")

### CELL 3 — Utility Functions

In [ ]:
def extract_frames(video_path, frame_skip=FRAME_SKIP, img_size=(IMG_HEIGHT, IMG_WIDTH)):
    cap = cv2.VideoCapture(video_path)
    frames, idx = [], 0
    if not cap.isOpened():
        print(f"  Warning: Cannot open {video_path}"); return [], 0, 0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    while True:
        ret, frame = cap.read()
        if not ret: break
        if idx % frame_skip == 0:
            frames.append(cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB),
                                     img_size).astype(np.float32))
        idx += 1
    cap.release()
    return frames, total, fps


def frames_to_sequences(frames, seq_length=SEQ_LENGTH, step=STEP_SIZE):
    seqs = []
    for s in range(0, len(frames) - seq_length + 1, step):
        seqs.append(frames[s:s+seq_length])
    return np.array(seqs, dtype=np.float32)

### CELL 4 — Build MobileNetV2 Feature Extractor

In [ ]:
def build_mobilenet_extractor():
    base = MobileNetV2(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
                       include_top=False, weights='imagenet')
    base.trainable = False
    inp = Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
    x   = preprocess_input(inp)
    x   = base(x, training=False)
    x   = GlobalAveragePooling2D()(x)
    return Model(inp, x, name='MobileNetV2_Extractor'), base

mobilenet_extractor, mobilenet_base = build_mobilenet_extractor()
mobilenet_extractor.summary()
print(f"\n✅ Feature dim per frame: {mobilenet_extractor.output_shape[-1]}")

### CELL 5 — Extract & Cache MobileNet Features

In [ ]:
def extract_and_cache_features(video_path, extractor, cache_dir=CACHE_DIR, batch_size=32):
    name  = os.path.splitext(os.path.basename(video_path))[0]
    cache = os.path.join(cache_dir, f"{name}_features.npy")
    if os.path.exists(cache): return np.load(cache)
    frames, _, _ = extract_frames(video_path)
    if len(frames) == 0: return np.array([])
    feats = np.vstack([
        extractor.predict(np.array(frames[i:i+batch_size]), verbose=0)
        for i in range(0, len(frames), batch_size)
    ])
    np.save(cache, feats)
    return feats


def features_to_sequences(feats, seq_length=SEQ_LENGTH, step=STEP_SIZE):
    seqs = []
    for s in range(0, len(feats) - seq_length + 1, step):
        seqs.append(feats[s:s+seq_length])
    return np.array(seqs, dtype=np.float32)


print('Extracting MobileNet features from TRAINING videos...')
train_feat_seqs, train_labels_raw = [], []
for vname in tqdm(train_videos, desc='Train Features'):
    feats = extract_and_cache_features(os.path.join(TRAIN_VIDEO_DIR, vname),
                                        mobilenet_extractor)
    seqs = features_to_sequences(feats)
    if len(seqs):
        train_feat_seqs.append(seqs)
        train_labels_raw.extend([0]*len(seqs))

X_train_raw = np.vstack(train_feat_seqs)
y_train_raw = np.array(train_labels_raw)
print(f"\n✅ Training sequences: {X_train_raw.shape}  (all Normal=0)")

### CELL 6 — Build Balanced Dataset (Normal + Synthetic Abnormal)

In [ ]:
def create_synthetic_abnormal(normal_seqs, noise_factor=2.5):
    n  = len(normal_seqs)
    ab = normal_seqs[np.random.choice(n, n)].copy()
    ab += np.random.normal(0, noise_factor * ab.std(), ab.shape)
    for i in range(len(ab)):
        ab[i] = ab[i][np.random.permutation(SEQ_LENGTH)]
    return ab

X_abn  = create_synthetic_abnormal(X_train_raw)
X_all  = np.vstack([X_train_raw, X_abn])
y_all  = np.hstack([np.zeros(len(X_train_raw)), np.ones(len(X_abn))]).astype(np.float32)
shuf   = np.random.permutation(len(X_all))
X_all, y_all = X_all[shuf], y_all[shuf]

X_tr, X_val, y_tr, y_val = train_test_split(
    X_all, y_all, test_size=VAL_SPLIT, random_state=42, stratify=y_all)

print(f"Train set : {X_tr.shape}  | Val set: {X_val.shape}")

### CELL 7 — Build LSTM + Attention + Sigmoid Model

In [ ]:
def build_lstm_model_with_attention(seq_len=SEQ_LENGTH, feat_dim=1280, lr=LEARNING_RATE):
    inp = Input(shape=(seq_len, feat_dim), name='feature_sequence')

    # LSTM-1: return_sequences=True so attention can weight each timestep
    x   = LSTM(256, return_sequences=True, dropout=0.3,
                recurrent_dropout=0.2, name='lstm_1')(inp)
    x   = Dropout(0.4)(x)                                         # (batch, 16, 256)

    # ── Attention layer (XAI-transparent temporal weighting) ──
    # e_t = tanh(W·h_t + b)   →   α_t = softmax(e_t)   →   context = Σ α_t·h_t
    attn_scores = Dense(1, activation='tanh', name='attn_score')(x)  # (batch, 16, 1)
    attn_weights = tf.keras.layers.Softmax(axis=1, name='attn_weights')(attn_scores)
    # Weighted sum over time → context vector
    context = tf.keras.layers.Multiply(name='attn_apply')([x, attn_weights])
    context = tf.keras.layers.Lambda(lambda z: tf.reduce_sum(z, axis=1),
                                     name='attn_context')(context)   # (batch, 256)

    # LSTM-2 on the context
    x   = tf.keras.layers.Reshape((1, 256))(context)
    x   = LSTM(128, return_sequences=False, dropout=0.2,
                recurrent_dropout=0.2, name='lstm_2')(x)
    x   = Dropout(0.3)(x)

    x   = Dense(64, activation='relu', name='dense_64')(x)
    x   = BatchNormalization(name='batch_norm')(x)
    x   = Dropout(0.2)(x)
    out = Dense(1, activation='sigmoid', name='anomaly_prob')(x)

    model = Model(inp, out, name='MobileNet_LSTM_Attention_Anomaly')
    model.compile(optimizer=Adam(lr), loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

    # Sub-model to extract attention weights (for XAI panel 5)
    attn_model = Model(inp, model.get_layer('attn_weights').output,
                       name='Attention_Extractor')
    return model, attn_model

lstm_model, attn_model = build_lstm_model_with_attention()
lstm_model.summary()
print(f"\n✅ Sigmoid output → prob > {THRESHOLD} = ABNORMAL")
print("✅ Attention sub-model built for XAI temporal explanations")

### CELL 8 — Train Model

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_auc', mode='max', patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4,
                      min_lr=1e-6, verbose=1),
    ModelCheckpoint(MODEL_PATH, monitor='val_auc', mode='max',
                    save_best_only=True, verbose=1)
]

print('Starting training...\n')
history = lstm_model.fit(
    X_tr, y_tr, validation_data=(X_val, y_val),
    batch_size=BATCH_SIZE, epochs=EPOCHS, callbacks=callbacks, verbose=1
)
print(f"\n✅ Best model saved → {MODEL_PATH}")

### CELL 9 — Training History Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training History — MobileNet + LSTM + Attention', fontsize=14, fontweight='bold')
for ax, metric, title in [
    (axes[0], 'loss',     'Loss'),
    (axes[1], 'accuracy', 'Accuracy'),
    (axes[2], 'auc',      'ROC-AUC')
]:
    ax.plot(history.history[metric],          lw=2, color='steelblue', label='Train')
    ax.plot(history.history[f'val_{metric}'], lw=2, color='tomato',    label='Val')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

### CELL 10 — Inference Function (Frame-Level Probability)

In [ ]:
def predict_frame_probabilities(video_path, extractor, model,
                                 seq_len=SEQ_LENGTH, step=STEP_SIZE, threshold=THRESHOLD):
    feats = extract_and_cache_features(video_path, extractor)
    if len(feats) < seq_len: return np.array([]), np.array([])

    seqs, starts = [], []
    for s in range(0, len(feats) - seq_len + 1, step):
        seqs.append(feats[s:s+seq_len]); starts.append(s)

    seq_probs = model.predict(np.array(seqs), verbose=0).flatten()

    accum = np.zeros(len(feats))
    count = np.zeros(len(feats))
    for i, s in enumerate(starts):
        accum[s:s+seq_len] += seq_probs[i]
        count[s:s+seq_len] += 1

    frame_probs  = accum / np.maximum(count, 1)
    frame_labels = (frame_probs > threshold).astype(int)
    return frame_probs, frame_labels

print('✅ Inference function ready.')

### CELL 11 — Predict on ALL Videos

In [ ]:
train_results, test_results = {}, {}

for vname in tqdm(train_videos, desc='Train Predict'):
    probs, labels = predict_frame_probabilities(
        os.path.join(TRAIN_VIDEO_DIR, vname), mobilenet_extractor, lstm_model)
    if len(probs) == 0: continue
    pct = 100.0 * labels.mean()
    train_results[vname] = {
        'probs': probs, 'labels': labels, 'pct_abnormal': pct,
        'mean_prob': probs.mean(), 'max_prob': probs.max(),
        'verdict': 'ABNORMAL' if pct > 10 else 'NORMAL'
    }

for vname in tqdm(test_videos, desc='Test Predict'):
    probs, labels = predict_frame_probabilities(
        os.path.join(TEST_VIDEO_DIR, vname), mobilenet_extractor, lstm_model)
    if len(probs) == 0: continue
    pct = 100.0 * labels.mean()
    test_results[vname] = {
        'probs': probs, 'labels': labels, 'pct_abnormal': pct,
        'mean_prob': probs.mean(), 'max_prob': probs.max(),
        'verdict': 'ABNORMAL' if pct > 10 else 'NORMAL'
    }

print(f"\n✅ Train: {len(train_results)} videos | Test: {len(test_results)} videos predicted")

---
## 🔍 XAI Section — Explainability Analysis

The following cells apply multiple XAI methods to explain the model's decisions.

---

### 🧠 XAI CELL 1 — GRAD-CAM: WHY did this frame get flagged?

In [ ]:
def build_gradcam_model(base_model):
    """
    Returns a model whose outputs are:
      [last_conv_feature_map, anomaly_score_through_GAP_and_rest_of_pipeline]
    We hook into the last Conv block of MobileNetV2.
    """
    last_conv_layer = base_model.get_layer('Conv_1')          # 7×7×1280
    grad_model = Model(
        inputs  = base_model.input,
        outputs = [last_conv_layer.output, base_model.output]
    )
    return grad_model

def compute_gradcam(frame_rgb, full_cnn_model, pred_index=0):
    """
    frame_rgb : np.ndarray (224,224,3) uint8 or float32
    Returns   : heatmap np.ndarray (224,224), values in [0,1]
    """
    img = preprocess_input(frame_rgb.copy().astype(np.float32))
    img = np.expand_dims(img, axis=0)                         # (1,224,224,3)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = full_cnn_model(img)
        loss = predictions[:, pred_index]                     # scalar anomaly score

    # Gradient of score w.r.t. last conv feature map
    grads = tape.gradient(loss, conv_outputs)                 # (1,7,7,1280)

    # Channel-wise importance weights (Global Average over spatial dims)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))     # (1280,)

    # Weight feature maps by their importance
    conv_outputs = conv_outputs[0]                            # (7,7,1280)
    cam = conv_outputs @ pooled_grads[..., tf.newaxis]        # (7,7,1)
    cam = tf.squeeze(cam)                                     # (7,7)
    cam = tf.nn.relu(cam)                                     # keep only positive
    cam = cam / (tf.math.reduce_max(cam) + 1e-8)             # normalise to [0,1]

    # Upsample to original frame size
    cam = cv2.resize(cam.numpy(), (IMG_WIDTH, IMG_HEIGHT))
    return cam

def overlay_gradcam(frame_rgb, cam, alpha=0.45):
    """Overlay Grad-CAM heatmap on original frame."""
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap  = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB).astype(np.float32)
    frame_f  = frame_rgb.astype(np.float32)
    superimposed = (frame_f * (1 - alpha) + heatmap * alpha).clip(0, 255).astype(np.uint8)
    return superimposed

# Build Grad-CAM model (CNN only — separate from LSTM)
gradcam_model = build_gradcam_model(mobilenet_extractor)
print("✅ Grad-CAM model built.")

# ── Apply Grad-CAM to MOST ANOMALOUS frame in each test video ──
print("\n" + "="*70)
print("  GRAD-CAM — Spatial Explanations for Anomalous Frames")
print("="*70)

n_test_show = min(6, len(test_videos))
fig, axes = plt.subplots(n_test_show, 3, figsize=(15, n_test_show * 4))
fig.suptitle('Grad-CAM: What spatial region triggered the anomaly?\n'
             'Left=Original | Middle=Grad-CAM heatmap | Right=Overlay',
             fontsize=13, fontweight='bold', y=1.01)

for row, vname in enumerate(list(test_results.keys())[:n_test_show]):
    # Find the frame with highest anomaly probability
    res         = test_results[vname]
    peak_frame  = int(np.argmax(res['probs']))

    frames, _, _ = extract_frames(os.path.join(TEST_VIDEO_DIR, vname))
    frame_idx    = min(peak_frame, len(frames)-1)
    frame_rgb    = frames[frame_idx].astype(np.uint8)

    cam           = compute_gradcam(frame_rgb, gradcam_model)
    overlay       = overlay_gradcam(frame_rgb, cam)

    # Column 0: Original frame
    axes[row,0].imshow(frame_rgb)
    axes[row,0].set_title(f'{vname} | Frame {frame_idx}', fontsize=9)
    axes[row,0].set_ylabel(f'P={res["probs"][peak_frame]:.3f}\n[{res["verdict"]}]',
                            fontsize=8, color='red' if res['verdict']=='ABNORMAL' else 'green')

    # Column 1: Raw Grad-CAM heatmap
    axes[row,1].imshow(cam, cmap='jet', vmin=0, vmax=1)
    axes[row,1].set_title('Grad-CAM (channel importance ×  feature maps)', fontsize=8)
    axes[row,1].set_xlabel('Warm=high activation → anomaly driver', fontsize=7)

    # Column 2: Overlay
    axes[row,2].imshow(overlay)
    axes[row,2].set_title('Frame + Grad-CAM overlay', fontsize=8)

    for ax in axes[row]: ax.set_xticks([]); ax.set_yticks([])
    for ax in axes[row,1:]: [s.set_edgecolor('#e74c3c'), s.set_linewidth(2)
                              for s in ax.spines.values()]

plt.tight_layout()
save_path = os.path.join(XAI_DIR, 'xai_gradcam.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Grad-CAM saved → {save_path}')

### 🧠 XAI CELL 2 — SALIENCY MAP (Vanilla Gradient)

In [ ]:
def compute_saliency(frame_rgb, cnn_model):
    """Vanilla gradient saliency map for a single frame."""
    img_tensor = tf.Variable(
        preprocess_input(frame_rgb.copy().astype(np.float32))[np.newaxis], dtype=tf.float32
    )
    with tf.GradientTape() as tape:
        preds = cnn_model(img_tensor)
        score = tf.reduce_max(preds)
    grads = tape.gradient(score, img_tensor)                  # (1,224,224,3)
    saliency = tf.reduce_max(tf.abs(grads[0]), axis=-1).numpy()  # collapse RGB
    saliency  = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
    return saliency

print("\n" + "="*70)
print("  SALIENCY MAPS — Pixel-level gradient sensitivity")
print("="*70)

n_show = min(4, len(test_videos))
fig, axes = plt.subplots(n_show, 2, figsize=(10, n_show * 3.5))
fig.suptitle('Vanilla Gradient Saliency: ∂output/∂pixel — which pixels are most influential?',
             fontsize=12, fontweight='bold')

for row, vname in enumerate(list(test_results.keys())[:n_show]):
    res        = test_results[vname]
    peak_frame = int(np.argmax(res['probs']))
    frames, _, _ = extract_frames(os.path.join(TEST_VIDEO_DIR, vname))
    frame_rgb  = frames[min(peak_frame, len(frames)-1)].astype(np.uint8)

    saliency = compute_saliency(frame_rgb, mobilenet_extractor)

    axes[row,0].imshow(frame_rgb)
    axes[row,0].set_title(f'{vname} — Original', fontsize=9)

    axes[row,1].imshow(saliency, cmap='hot')
    axes[row,1].set_title(f'Saliency | P={res["probs"][peak_frame]:.3f}', fontsize=9)
    axes[row,1].set_xlabel('Bright = pixel most responsible for prediction', fontsize=7)

    for ax in axes[row]: ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
save_path = os.path.join(XAI_DIR, 'xai_saliency.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Saliency maps saved → {save_path}')

### 🧠 XAI CELL 3 — SHAP (SHapley Additive exPlanations)

In [ ]:
print("\n" + "="*70)
print("  SHAP — Feature Attribution on 1280-dim MobileNet Features")
print("="*70)

# Background = sample of normal (training) sequences
N_BACKGROUND = 100
background_idx = np.random.choice(len(X_tr[y_tr == 0]), N_BACKGROUND, replace=False)
background_seqs = X_tr[y_tr == 0][background_idx]            # (100, 16, 1280)

# Sequences to explain = highest-probability test sequences
test_seq_list, test_seq_labels = [], []
for vname in list(test_results.keys())[:5]:
    feats = extract_and_cache_features(
        os.path.join(TEST_VIDEO_DIR, vname), mobilenet_extractor)
    seqs  = features_to_sequences(feats)
    probs = lstm_model.predict(seqs, verbose=0).flatten()
    top_idx = np.argsort(probs)[-3:]                          # top-3 most anomalous
    test_seq_list.extend(seqs[top_idx])
    test_seq_labels.extend(probs[top_idx])

explain_seqs  = np.array(test_seq_list[:10])                  # (10, 16, 1280)
explain_probs = np.array(test_seq_labels[:10])

# DeepExplainer
explainer  = shap.DeepExplainer(lstm_model, background_seqs)
shap_vals  = explainer.shap_values(explain_seqs)              # list[(10, 16, 1280)]
shap_array = shap_vals[0]                                     # (10, 16, 1280)

# Mean |SHAP| across sequences and timesteps → per feature importance
feat_importance = np.abs(shap_array).mean(axis=(0, 1))       # (1280,)
top_features    = np.argsort(feat_importance)[-30:][::-1]

# ── Plot 1: Top-30 feature importances (bar chart) ──
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].barh(range(30), feat_importance[top_features],
             color=['#e74c3c' if feat_importance[f] > feat_importance[top_features].mean()
                    else '#3498db' for f in top_features])
axes[0].set_yticks(range(30))
axes[0].set_yticklabels([f'Dim {f}' for f in top_features], fontsize=8)
axes[0].set_xlabel('Mean |SHAP value|  (higher = more influential)')
axes[0].set_title('Top 30 MobileNet Feature Dims by SHAP Importance\n'
                  'φ_i = Σ [|S|!(|F|-|S|-1)!/|F|!]·[f(S∪{i})−f(S)]',
                  fontsize=11, fontweight='bold')
axes[0].invert_yaxis()
axes[0].axvline(feat_importance[top_features].mean(), color='black',
                ls='--', lw=1.5, label='mean importance')
axes[0].legend()
axes[0].grid(axis='x', alpha=0.3)

# ── Plot 2: SHAP force decomposition for one sequence ──
seq_idx = 0
shap_frame_mean = shap_array[seq_idx].mean(axis=0)           # mean over 16 timesteps
top_k = 20
top_idx_plot = np.argsort(np.abs(shap_frame_mean))[-top_k:][::-1]

colors_bar = ['#e74c3c' if shap_frame_mean[i] > 0 else '#2ecc71'
              for i in top_idx_plot]
axes[1].barh(range(top_k), shap_frame_mean[top_idx_plot], color=colors_bar)
axes[1].set_yticks(range(top_k))
axes[1].set_yticklabels([f'Dim {i}' for i in top_idx_plot], fontsize=8)
axes[1].axvline(0, color='black', lw=1)
axes[1].set_xlabel('SHAP value  (red=pushes ABNORMAL | green=pushes NORMAL)')
axes[1].set_title(f'SHAP Force Plot — Sequence (P={explain_probs[seq_idx]:.3f})\n'
                  f'Base E[f(x)]≈0.50  |  Prediction={explain_probs[seq_idx]:.3f}  '
                  f'|  Σφ_i ≈ {explain_probs[seq_idx]-0.50:.3f}',
                  fontsize=11, fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
save_path = os.path.join(XAI_DIR, 'xai_shap.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ SHAP plots saved → {save_path}')

# ── Plot 3: SHAP temporal heatmap (16 frames × top features) ──
fig, ax = plt.subplots(figsize=(14, 6))
shap_temporal = shap_array[seq_idx][:, top_features[:20]]    # (16, 20)
sns.heatmap(shap_temporal.T, ax=ax, cmap='RdYlGn_r',
            xticklabels=[f't{i}' for i in range(SEQ_LENGTH)],
            yticklabels=[f'Dim {f}' for f in top_features[:20]],
            center=0, linewidths=0.3, cbar_kws={'label': 'SHAP value'})
ax.set_xlabel('Timestep (frame index in 16-frame window)', fontsize=11)
ax.set_ylabel('MobileNet Feature Dimension', fontsize=11)
ax.set_title('SHAP Temporal Heatmap — Which feature at which timestep drove the prediction?\n'
             'Red = pushes ABNORMAL | Green = pushes NORMAL',
             fontsize=12, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(XAI_DIR, 'xai_shap_temporal.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ SHAP temporal heatmap saved → {save_path}')

### 🧠 XAI CELL 4 — LIME (Local Interpretable Model-Agnostic Explanations)

In [ ]:
def make_lime_predictor(extractor, model, seq_len=SEQ_LENGTH):
    """
    LIME calls this with batches of perturbed images.
    We extract features → replicate to seq_len → run LSTM → return probability.
    This is the "black box" LIME queries.
    """
    def predictor(images):                                    # images: (N,224,224,3)
        probs = []
        for img in images:
            feat = extractor.predict(img[np.newaxis].astype(np.float32), verbose=0)  # (1,1280)
            seq  = np.tile(feat, (seq_len, 1))[np.newaxis]   # (1,16,1280)
            p    = model.predict(seq, verbose=0)[0, 0]
            probs.append([1-p, p])                            # [normal_prob, anomaly_prob]
        return np.array(probs)
    return predictor

lime_predictor = make_lime_predictor(mobilenet_extractor, lstm_model)
lime_explainer = lime_image.LimeImageExplainer()

print("\n" + "="*70)
print("  LIME — Superpixel Importance via Local Perturbation")
print("="*70)

n_lime = min(4, len(test_videos))
fig, axes = plt.subplots(n_lime, 3, figsize=(15, n_lime * 4))
fig.suptitle('LIME: Which image regions drive the anomaly prediction?\n'
             'Green superpixels = support ABNORMAL | Red = suppresses anomaly',
             fontsize=12, fontweight='bold', y=1.01)

for row, vname in enumerate(list(test_results.keys())[:n_lime]):
    res   = test_results[vname]
    peak  = int(np.argmax(res['probs']))
    frames, _, _ = extract_frames(os.path.join(TEST_VIDEO_DIR, vname))
    frame_rgb = frames[min(peak, len(frames)-1)].astype(np.uint8)

    # Run LIME (num_samples controls accuracy; higher → more stable but slower)
    explanation = lime_explainer.explain_instance(
        frame_rgb,
        lime_predictor,
        top_labels=1,
        hide_color=0,
        num_samples=500,
        num_features=10
    )

    # Get image + mask for the ABNORMAL class (label=1)
    temp_img, mask = explanation.get_image_and_mask(
        label=1, positive_only=False, num_features=8, hide_rest=False)

    axes[row,0].imshow(frame_rgb)
    axes[row,0].set_title(f'{vname} — Original (P={res["probs"][peak]:.3f})', fontsize=9)

    axes[row,1].imshow(mark_boundaries(temp_img, mask))
    axes[row,1].set_title('LIME Superpixel boundaries', fontsize=9)

    # Show only positive (anomaly-supporting) regions
    temp_pos, mask_pos = explanation.get_image_and_mask(
        label=1, positive_only=True, num_features=5, hide_rest=True)
    axes[row,2].imshow(mark_boundaries(temp_pos, mask_pos))
    axes[row,2].set_title('Top 5 anomaly-positive superpixels', fontsize=9)
    axes[row,2].set_xlabel('Only shown regions drove P toward ABNORMAL', fontsize=7)

    for ax in axes[row]: ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
save_path = os.path.join(XAI_DIR, 'xai_lime.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ LIME explanations saved → {save_path}')

### 🧠 XAI CELL 5 — ATTENTION WEIGHTS: Which FRAME in the sequence mattered most?

In [ ]:
print("\n" + "="*70)
print("  ATTENTION WEIGHTS — Temporal Importance per Frame in Sequence")
print("="*70)

n_attn = min(6, len(test_videos))
fig, axes = plt.subplots(n_attn, 1, figsize=(16, n_attn * 3))
fig.suptitle('LSTM Attention Weights α_t = softmax(tanh(W_a·h_t+b_a))\n'
             'Higher bar = LSTM relied on this timestep more for its anomaly decision',
             fontsize=12, fontweight='bold', y=1.01)

for row, vname in enumerate(list(test_results.keys())[:n_attn]):
    res  = test_results[vname]
    feats = extract_and_cache_features(
        os.path.join(TEST_VIDEO_DIR, vname), mobilenet_extractor)

    # Find the window with highest anomaly probability
    seqs, starts = [], []
    for s in range(0, len(feats) - SEQ_LENGTH + 1, STEP_SIZE):
        seqs.append(feats[s:s+SEQ_LENGTH]); starts.append(s)
    seqs_arr = np.array(seqs)
    seq_probs = lstm_model.predict(seqs_arr, verbose=0).flatten()
    best_seq_idx = np.argmax(seq_probs)

    # Extract attention weights for the most anomalous sequence
    best_seq = seqs_arr[best_seq_idx:best_seq_idx+1]           # (1, 16, 1280)
    attn_w   = attn_model.predict(best_seq, verbose=0)[0]     # (16, 1)
    attn_w   = attn_w.flatten()                               # (16,)

    ax = axes[row]
    timesteps = np.arange(SEQ_LENGTH)
    colors = ['#e74c3c' if a > attn_w.mean() else '#3498db' for a in attn_w]
    bars = ax.bar(timesteps, attn_w, color=colors, edgecolor='white', linewidth=0.5)
    ax.axhline(1/SEQ_LENGTH, color='black', ls='--', lw=1, alpha=0.6,
               label=f'Uniform baseline (1/16 = {1/SEQ_LENGTH:.3f})')
    ax.axhline(attn_w.mean(), color='orange', ls=':', lw=1.5,
               label=f'Mean α = {attn_w.mean():.3f}')

    # Annotate peak
    peak_t = np.argmax(attn_w)
    ax.annotate(f'Peak\nt={peak_t}\nα={attn_w[peak_t]:.3f}',
                xy=(peak_t, attn_w[peak_t]),
                xytext=(peak_t + 0.5, attn_w[peak_t] + 0.005),
                fontsize=7, color='#e74c3c',
                arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1))

    verdict_color = 'red' if res['verdict'] == 'ABNORMAL' else 'green'
    ax.set_title(f'{vname}  |  P={seq_probs[best_seq_idx]:.3f}  [{res["verdict"]}]  '
                 f'Window start frame: {starts[best_seq_idx]}',
                 fontsize=9, fontweight='bold', color=verdict_color)
    ax.set_xlabel('Timestep t within 16-frame window', fontsize=8)
    ax.set_ylabel('α_t  (attention weight)', fontsize=8)
    ax.set_xticks(timesteps)
    ax.set_xticklabels([f't{i}' for i in timesteps], fontsize=7)
    ax.set_ylim(0, max(attn_w) * 1.25)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
save_path = os.path.join(XAI_DIR, 'xai_attention_weights.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Attention weights plot saved → {save_path}')

### 🧠 XAI CELL 6 — LSTM GATE ANALYSIS: What did the LSTM remember/forget?

In [ ]:
def build_gate_extractor(lstm_model):
    """
    Creates models to extract LSTM-1 gate activations at each timestep.
    We replicate the LSTM computation manually using its weights.
    """
    lstm1_layer = lstm_model.get_layer('lstm_1')
    W, U, b = lstm1_layer.get_weights()          # kernel(1280,1024), recurrent(256,1024), bias(1024,)
    units   = 256
    # Gates: i(0:256), f(256:512), c(512:768), o(768:1024)
    W_i, W_f, W_c, W_o = W[:,:units], W[:,units:2*units], W[:,2*units:3*units], W[:,3*units:]
    U_i, U_f, U_c, U_o = U[:,:units], U[:,units:2*units], U[:,2*units:3*units], U[:,3*units:]
    b_i, b_f, b_c, b_o = b[:units], b[units:2*units], b[2*units:3*units], b[3*units:]
    return (W_i,W_f,W_c,W_o, U_i,U_f,U_c,U_o, b_i,b_f,b_c,b_o)

def compute_gate_activations(sequence, gate_weights):
    """
    Manually unrolls LSTM-1 to get gate values at each timestep.
    sequence: (16, 1280)
    Returns dict of gate arrays each shape (16, 256)
    """
    W_i,W_f,W_c,W_o, U_i,U_f,U_c,U_o, b_i,b_f,b_c,b_o = gate_weights
    h = np.zeros((256,)); c = np.zeros((256,))
    gates = {'input':[], 'forget':[], 'cell':[], 'output':[]}
    for t in range(len(sequence)):
        x = sequence[t]
        i_t = 1/(1+np.exp(-(x @ W_i + h @ U_i + b_i)))
        f_t = 1/(1+np.exp(-(x @ W_f + h @ U_f + b_f)))
        g_t = np.tanh(x @ W_c + h @ U_c + b_c)
        o_t = 1/(1+np.exp(-(x @ W_o + h @ U_o + b_o)))
        c   = f_t * c + i_t * g_t
        h   = o_t * np.tanh(c)
        gates['input'].append(i_t); gates['forget'].append(f_t)
        gates['cell'].append(g_t);  gates['output'].append(o_t)
    return {k: np.array(v) for k,v in gates.items()}       # each (16,256)

print("\n" + "="*70)
print("  LSTM GATE ANALYSIS — Internal Memory Transparency")
print("="*70)

gate_weights = build_gate_extractor(lstm_model)
n_gate = min(3, len(test_videos))

fig, axes = plt.subplots(n_gate, 4, figsize=(20, n_gate * 3.5))
fig.suptitle('LSTM Gate Activations — Mean over 256 units per timestep\n'
             'Forget≈0: LSTM reset memory  |  Input≈1: new anomalous pattern written in',
             fontsize=12, fontweight='bold', y=1.01)

for row, vname in enumerate(list(test_results.keys())[:n_gate]):
    feats = extract_and_cache_features(
        os.path.join(TEST_VIDEO_DIR, vname), mobilenet_extractor)
    seqs = features_to_sequences(feats)
    probs = lstm_model.predict(seqs, verbose=0).flatten()
    best  = np.argmax(probs)
    gates = compute_gate_activations(seqs[best], gate_weights)

    t = np.arange(SEQ_LENGTH)
    gate_colors = {'input':'#e74c3c', 'forget':'#3498db', 'cell':'#f39c12', 'output':'#2ecc71'}
    for col, (gname, gcolor) in enumerate(gate_colors.items()):
        ax = axes[row, col]
        mean_gate = gates[gname].mean(axis=1)                 # mean over 256 units
        ax.plot(t, mean_gate, color=gcolor, lw=2, marker='o', markersize=4)
        ax.fill_between(t, 0, mean_gate, alpha=0.15, color=gcolor)
        ax.axhline(mean_gate.mean(), color='black', ls='--', lw=1,
                   alpha=0.6, label=f'mean={mean_gate.mean():.3f}')
        ax.set_ylim(0, 1)
        ax.set_title(f'{gname.capitalize()} gate\n{vname[:15]}' if row==0
                     else f'{gname.capitalize()} gate', fontsize=9, fontweight='bold')
        ax.set_xlabel('Timestep t', fontsize=8)
        ax.set_ylabel(f'σ({gname[0]}_t) mean', fontsize=7)
        ax.set_xticks(t); ax.set_xticklabels([f't{i}' for i in t], fontsize=6)
        ax.legend(fontsize=7); ax.grid(alpha=0.3)

        # Highlight peak
        peak = np.argmax(mean_gate) if gname in ['input','output'] else np.argmin(mean_gate)
        ax.axvline(peak, color=gcolor, ls=':', lw=2, alpha=0.7)

plt.tight_layout()
save_path = os.path.join(XAI_DIR, 'xai_lstm_gates.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ LSTM gate analysis saved → {save_path}')

### 🧠 XAI CELL 7 — FEATURE SPACE VISUALISATION (t-SNE / UMAP)

In [ ]:
from sklearn.manifold import TSNE

print("\n" + "="*70)
print("  t-SNE FEATURE SPACE — Do normal/abnormal separate in representation space?")
print("="*70)

# Use val set (has both normal and abnormal)
N_TSNE    = min(2000, len(X_val))
idx_tsne  = np.random.choice(len(X_val), N_TSNE, replace=False)
X_tsne_in = X_val[idx_tsne].mean(axis=1)                     # mean over 16 timesteps (N,1280)
y_tsne    = y_val[idx_tsne]

tsne      = TSNE(n_components=2, perplexity=40, n_iter=1000, random_state=42, verbose=1)
X_2d      = tsne.fit_transform(X_tsne_in)                    # (N, 2)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Scatter by class
sc0 = axes[0].scatter(X_2d[y_tsne==0, 0], X_2d[y_tsne==0, 1],
                       c='#3498db', alpha=0.4, s=12, label='Normal', edgecolors='none')
sc1 = axes[0].scatter(X_2d[y_tsne==1, 0], X_2d[y_tsne==1, 1],
                       c='#e74c3c', alpha=0.6, s=12, label='Abnormal', edgecolors='none')
axes[0].set_title('t-SNE of 1280-dim MobileNet Features\nColoured by Ground Truth Label',
                   fontsize=12, fontweight='bold')
axes[0].set_xlabel('t-SNE dim 1'); axes[0].set_ylabel('t-SNE dim 2')
axes[0].legend(fontsize=10); axes[0].grid(alpha=0.2)

# Scatter by LSTM predicted probability
probs_tsne = lstm_model.predict(X_val[idx_tsne], verbose=0).flatten()
sc2 = axes[1].scatter(X_2d[:,0], X_2d[:,1], c=probs_tsne,
                       cmap='RdYlGn_r', alpha=0.5, s=12, edgecolors='none',
                       vmin=0, vmax=1)
plt.colorbar(sc2, ax=axes[1], label='Anomaly probability P(abnormal)')
axes[1].set_title('Same t-SNE — coloured by LSTM anomaly probability\n'
                   'Gradient shows how well LSTM separates the feature space',
                   fontsize=12, fontweight='bold')
axes[1].set_xlabel('t-SNE dim 1'); axes[1].set_ylabel('t-SNE dim 2')
axes[1].grid(alpha=0.2)

plt.tight_layout()
save_path = os.path.join(XAI_DIR, 'xai_tsne.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ t-SNE plot saved → {save_path}')

### 🧠 XAI CELL 8 — DECISION BOUNDARY: Threshold sensitivity analysis

In [ ]:
from sklearn.metrics import precision_recall_curve

print("\n" + "="*70)
print("  THRESHOLD SENSITIVITY — XAI for the decision rule p > 0.5")
print("="*70)

val_probs = lstm_model.predict(X_val, verbose=0).flatten()
fpr, tpr, roc_thresholds = roc_curve(y_val, val_probs)
auc_val = roc_auc_score(y_val, val_probs)
precision, recall, pr_thresholds = precision_recall_curve(y_val, val_probs)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('XAI: Decision Rule Transparency — How does threshold choice affect performance?',
             fontsize=13, fontweight='bold')

# ROC Curve with operating point
axes[0].plot(fpr, tpr, lw=2.5, color='darkorange', label=f'AUC={auc_val:.4f}')
axes[0].plot([0,1],[0,1],'k--',lw=1, label='Random classifier')
# Mark current threshold
idx_thresh = np.argmin(np.abs(roc_thresholds - THRESHOLD))
axes[0].scatter(fpr[idx_thresh], tpr[idx_thresh], s=120, color='red', zorder=5,
                label=f'Current threshold={THRESHOLD}')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve\nOperating point at threshold=0.5', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Precision-Recall Curve
axes[1].plot(recall, precision, lw=2.5, color='steelblue')
idx_pr = np.argmin(np.abs(pr_thresholds - THRESHOLD)) if len(pr_thresholds) > 0 else 0
axes[1].scatter(recall[idx_pr], precision[idx_pr], s=120, color='red', zorder=5,
                label=f'threshold={THRESHOLD}')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve\nChoosing threshold = trading recall vs precision',
                   fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

# Probability distribution: normal vs abnormal
axes[2].hist(val_probs[y_val==0], bins=50, alpha=0.6, color='#3498db',
             density=True, label='Normal sequences')
axes[2].hist(val_probs[y_val==1], bins=50, alpha=0.6, color='#e74c3c',
             density=True, label='Abnormal sequences')
axes[2].axvline(THRESHOLD, color='black', lw=2, ls='--', label=f'Threshold={THRESHOLD}')
overlap_zone = axes[2].axvspan(0.35, 0.65, alpha=0.08, color='orange',
                                label='Uncertainty zone (~0.35-0.65)')
axes[2].set_xlabel('Anomaly probability P(abnormal)')
axes[2].set_ylabel('Density')
axes[2].set_title('Probability Distribution per Class\nShows separation quality + uncertainty zone',
                   fontsize=11, fontweight='bold')
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

plt.tight_layout()
save_path = os.path.join(XAI_DIR, 'xai_decision_boundary.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Decision boundary analysis saved → {save_path}')

### 🧠 XAI CELL 9 — SYNTHETIC ABNORMAL AUDIT

In [ ]:
print("\n" + "="*70)
print("  SYNTHETIC ABNORMAL AUDIT — Did the model learn real or fake anomaly patterns?")
print("="*70)

# SHAP for synthetic abnormal (from val set)
syn_abn_seqs = X_val[y_val == 1][:20]
real_abn_seqs = np.array(test_seq_list[:20]) if test_seq_list else syn_abn_seqs

shap_syn  = np.abs(explainer.shap_values(syn_abn_seqs[:10])[0]).mean(axis=(0,1))
shap_real = np.abs(explainer.shap_values(real_abn_seqs[:10])[0]).mean(axis=(0,1))

# Top features from each
top_syn  = np.argsort(shap_syn)[-20:][::-1]
top_real = np.argsort(shap_real)[-20:][::-1]
overlap  = len(set(top_syn) & set(top_real))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'SYNTHETIC vs REAL Anomaly SHAP Audit\n'
             f'Top-20 feature overlap: {overlap}/20  '
             f'({"✅ Good generalisation" if overlap>=10 else "⚠️ Possible overfit to synthetic"})',
             fontsize=12, fontweight='bold')

for ax, vals, top_idx, title, color in [
    (axes[0], shap_syn,  top_syn,  'Synthetic abnormal (training noise+shuffle)', '#f39c12'),
    (axes[1], shap_real, top_real, 'Real test anomalies (actual video)',          '#e74c3c')
]:
    ax.barh(range(20), vals[top_idx], color=color, alpha=0.8)
    ax.set_yticks(range(20))
    ax.set_yticklabels([f'Dim {i}' for i in top_idx], fontsize=8)
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.invert_yaxis(); ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
save_path = os.path.join(XAI_DIR, 'xai_synthetic_audit.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Synthetic audit plot saved → {save_path}')

---
## 🔍 XAI Section — Explainability Analysis

The following cells apply multiple XAI methods to explain the model's decisions.

---

### 🧠 XAI CELL 10 — COMPREHENSIVE XAI SUMMARY DASHBOARD

In [ ]:
fig = plt.figure(figsize=(22, 14))
fig.suptitle('Avenue Dataset — Explainable AI Summary Dashboard\n'
             'MobileNetV2 → Attention LSTM → Sigmoid', fontsize=16, fontweight='bold', y=1.01)

# ── Panel A: Pipeline with XAI hooks ──
ax_pipe = fig.add_axes([0.02, 0.72, 0.96, 0.22])
ax_pipe.set_xlim(0, 10); ax_pipe.set_ylim(0, 1); ax_pipe.axis('off')
stages = [
    ('Raw\nframe\n(224×224×3)', 0.3, '#ecf0f1'),
    ('MobileNetV2\nFeatures\n(1280-d)', 1.7, '#d5e8f3'),
    ('Sequence\nwindow\n(16×1280)', 3.1, '#d5e8f3'),
    ('LSTM-1\n(256 units)', 4.5, '#fde8cc'),
    ('Attention\nWeights α_t', 5.9, '#fde8cc'),
    ('LSTM-2\n(128 units)', 7.3, '#fde8cc'),
    ('Sigmoid\nP(abnormal)', 8.7, '#f9d0d0'),
]
xai_labels = {
    1: ('Grad-CAM\nSaliency', '#2980b9'),
    2: ('SHAP\nt-SNE',        '#2980b9'),
    3: ('SHAP\nGate analysis','#e67e22'),
    4: ('Attention\nweights',  '#e67e22'),
    5: ('Threshold\nsensitivity','#c0392b'),
}
for (label, x, col), (i, _) in zip(stages, enumerate(stages)):
    rect = plt.Rectangle((x-0.25, 0.15), 1.1, 0.65, color=col,
                          ec='#bdc3c7', lw=1, transform=ax_pipe.transData, zorder=2)
    ax_pipe.add_patch(rect)
    ax_pipe.text(x+0.3, 0.47, label, ha='center', va='center',
                 fontsize=7.5, fontweight='bold', zorder=3)
    if i < len(stages)-1:
        ax_pipe.annotate('', xy=(x+1.35+0.25-0.25, 0.47), xytext=(x+1.1, 0.47),
                          arrowprops=dict(arrowstyle='->', color='#7f8c8d', lw=1.5))
    if i in xai_labels:
        xai_text, xai_col = xai_labels[i]
        ax_pipe.text(x+0.3, 0.05, xai_text, ha='center', va='center',
                     fontsize=7, color=xai_col, style='italic')
        ax_pipe.plot([x+0.3, x+0.3], [0.15, 0.13], color=xai_col, lw=1, ls='--')

ax_pipe.set_title('Pipeline with XAI Instrumentation Points',
                   fontsize=10, fontweight='bold', pad=5)

# ── Panel B: Class Probability Distribution (compact) ──
ax_dist = fig.add_axes([0.02, 0.38, 0.30, 0.28])
ax_dist.hist(val_probs[y_val==0], bins=40, alpha=0.6, color='#3498db',
              density=True, label='Normal')
ax_dist.hist(val_probs[y_val==1], bins=40, alpha=0.6, color='#e74c3c',
              density=True, label='Abnormal')
ax_dist.axvline(0.5, color='black', lw=2, ls='--', label='T=0.5')
ax_dist.set_title('Probability Separation\n(Sigmoid output distribution)',
                   fontsize=9, fontweight='bold')
ax_dist.set_xlabel('P(abnormal)'); ax_dist.set_ylabel('Density')
ax_dist.legend(fontsize=8); ax_dist.grid(alpha=0.3)

# ── Panel C: Top SHAP features ──
ax_shap = fig.add_axes([0.36, 0.38, 0.30, 0.28])
top20 = np.argsort(feat_importance)[-15:][::-1]
ax_shap.barh(range(15), feat_importance[top20],
              color=['#e74c3c' if v > feat_importance[top20].mean() else '#3498db'
                     for v in feat_importance[top20]])
ax_shap.set_yticks(range(15))
ax_shap.set_yticklabels([f'Dim {d}' for d in top20], fontsize=7)
ax_shap.set_title('Top 15 SHAP Feature Importances\n(which MobileNet dims drive decisions)',
                   fontsize=9, fontweight='bold')
ax_shap.set_xlabel('Mean |SHAP value|')
ax_shap.invert_yaxis(); ax_shap.grid(axis='x', alpha=0.3)

# ── Panel D: ROC Curve ──
ax_roc = fig.add_axes([0.70, 0.38, 0.28, 0.28])
ax_roc.plot(fpr, tpr, lw=2.5, color='darkorange', label=f'AUC={auc_val:.4f}')
ax_roc.plot([0,1],[0,1],'k--',lw=1)
ax_roc.fill_between(fpr, tpr, alpha=0.1, color='darkorange')
ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
ax_roc.set_title(f'ROC Curve\nAUC = {auc_val:.4f}', fontsize=9, fontweight='bold')
ax_roc.legend(fontsize=9); ax_roc.grid(alpha=0.3)

# ── Panel E: XAI Methods Summary Table ──
ax_tbl = fig.add_axes([0.02, 0.02, 0.96, 0.32])
ax_tbl.axis('off')
table_data = [
    ['XAI Method',        'What it explains',                     'Input → Output',         'Math core',                          'Avenue insight'],
    ['Grad-CAM',          'WHICH PIXELS drive the CNN feature',   'Frame (224×224×3) → Heatmap (224×224)',
     'α_k = (1/Z)ΣΣ ∂y/∂A^k\nL_CAM = ReLU(Σ_k α_k·A^k)',  'Running figures, thrown objects appear as hot regions'],
    ['Saliency map',      'PIXEL sensitivity to prediction',      'Frame → |∂output/∂pixel|', '|∂y^c/∂x_{ij}| per pixel',        'Edges of moving persons have largest gradient magnitude'],
    ['SHAP',              'WHICH of 1280 MobileNet dims matter',  'Feature seq → φ_i per dim','Σφ_i = f(x)−E[f(x)]\nShapley values','Dims encoding motion blur & horizontal edges dominate'],
    ['LIME',              'WHICH image regions are causal',        'Frame → superpixel weights','g*=argmin Σπ(z)[f(z)−g(z\')]²',   'Central-scene pedestrian regions drive abnormal verdict'],
    ['Attention weights', 'WHICH timestep the LSTM focused on',   'Seq (16×1280) → α_t (16,)', 'α_t = softmax(tanh(W_a·h_t+b_a))','Peak attention at the frame where motion deviates most'],
    ['LSTM gate analysis','WHAT the LSTM remembered/forgot',      'Seq → gate values (16,256)','f_t⊙c_{t-1}+i_t⊙g_t=c_t',        'Forget gate drops at anomaly onset — LSTM resets context'],
    ['t-SNE',             'DO normal/abnormal cluster separately', '1280-d features → 2-d viz','KL(P||Q) minimisation',             'Two clear clusters confirm model learns discriminative repr.'],
    ['Threshold analysis','HOW SENSITIVE is the decision rule',   'P(abn) vs threshold → metrics','P-R / ROC sweep',                'Uncertainty zone 0.35–0.65 reveals borderline frames'],
]
col_widths = [0.14, 0.18, 0.20, 0.24, 0.24]
row_colors = [['#2c3e50']*5] + [['#f8f9fa','#ffffff'][r%2]]*1 * 8
header_color = '#2c3e50'
row_h = 0.1

for r, row in enumerate(table_data):
    for c, (cell, cw) in enumerate(zip(row, col_widths)):
        x_start = sum(col_widths[:c])
        bg = header_color if r == 0 else ('#f0f4f8' if r % 2 == 0 else 'white')
        fc = 'white' if r == 0 else 'black'
        rect = plt.Rectangle((x_start, 1 - (r+1)*row_h), cw, row_h,
                               transform=ax_tbl.transAxes, color=bg, ec='#dee2e6', lw=0.5,
                               clip_on=False, zorder=1)
        ax_tbl.add_patch(rect)
        ax_tbl.text(x_start + cw/2, 1 - (r+0.5)*row_h, cell,
                     ha='center', va='center', fontsize=6.5, color=fc,
                     fontweight='bold' if r==0 else 'normal',
                     transform=ax_tbl.transAxes, zorder=2, wrap=True,
                     multialignment='center')

ax_tbl.set_title('XAI Methods Applied — What each technique explains about this pipeline',
                  fontsize=10, fontweight='bold', pad=10)

plt.savefig(os.path.join(XAI_DIR, 'xai_summary_dashboard.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ XAI Summary Dashboard saved → {os.path.join(XAI_DIR, "xai_summary_dashboard.png")}')

### CELL 12 — Confusion Matrix + ROC (standard evaluation)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

val_preds = (val_probs > THRESHOLD).astype(int)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(fpr, tpr, lw=2.5, color='darkorange', label=f'AUC={auc_val:.4f}')
axes[0].plot([0,1],[0,1],'k--',lw=1)
axes[0].set_xlabel('FPR',fontsize=12); axes[0].set_ylabel('TPR',fontsize=12)
axes[0].set_title('ROC Curve — Validation Set',fontsize=13,fontweight='bold')
axes[0].legend(fontsize=12); axes[0].grid(alpha=0.3)

sns.heatmap(confusion_matrix(y_val, val_preds), annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Normal','Abnormal'], yticklabels=['Normal','Abnormal'])
axes[1].set_xlabel('Predicted',fontsize=12); axes[1].set_ylabel('Actual',fontsize=12)
axes[1].set_title('Confusion Matrix',fontsize=13,fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR,'roc_confusion.png'),dpi=150,bbox_inches='tight')
plt.show()
print(classification_report(y_val, val_preds, target_names=['Normal','Abnormal']))
print(f'ROC AUC: {auc_val:.4f}')

### CELL 13 — Save Model & XAI File Summary

In [ ]:
lstm_model.save(MODEL_PATH)
attn_model.save(os.path.join(BASE_DIR, 'attention_extractor.keras'))

print('\n' + '='*70)
print('  PIPELINE + XAI COMPLETE')
print('='*70)
print(f'  MobileNetV2    → 1280-dim features per frame (frozen)')
print(f'  LSTM-1 (256)   → Temporal sequence modelling')
print(f'  Attention      → Learnable temporal focus (XAI-native)')
print(f'  LSTM-2 (128)   → Compressed representation')
print(f'  Sigmoid        → Anomaly probability in [0, 1]')
print(f'  Threshold      → {THRESHOLD}   (prob > {THRESHOLD} = ABNORMAL)')
print('='*70)
print(f'\n  XAI Outputs saved to: {XAI_DIR}')
xai_files = [
    ('xai_gradcam.png',           'Grad-CAM spatial heatmaps'),
    ('xai_saliency.png',          'Vanilla gradient saliency maps'),
    ('xai_shap.png',              'SHAP feature importances + force plot'),
    ('xai_shap_temporal.png',     'SHAP temporal heatmap (16 frames × features)'),
    ('xai_lime.png',              'LIME superpixel explanations'),
    ('xai_attention_weights.png', 'LSTM attention weights per timestep'),
    ('xai_lstm_gates.png',        'LSTM gate activations (forget/input/cell/output)'),
    ('xai_tsne.png',              't-SNE feature space visualisation'),
    ('xai_decision_boundary.png', 'Threshold sensitivity analysis'),
    ('xai_synthetic_audit.png',   'Synthetic vs real anomaly SHAP audit'),
    ('xai_summary_dashboard.png', 'Comprehensive XAI summary dashboard'),
]
for fname, desc in xai_files:
    fpath = os.path.join(XAI_DIR, fname)
    status = '✅' if os.path.exists(fpath) else '⏳'
    print(f'  {status}  {fname:<35}  {desc}')
print('='*70)